# **Implementation of a 512 bits randomized primality test**

Advanced Algorithms and Parallel Programming challenge 1

*Andrea Bellani*

# **Notebook setup**

**Download the code**

In [18]:
!git clone https://github.com/google/benchmark.git
!git clone https://github.com/google/googletest.git benchmark/googletest

Cloning into 'benchmark'...
remote: Enumerating objects: 9897, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 9897 (delta 165), reused 106 (delta 106), pack-reused 9674 (from 3)
Receiving objects: 100% (9897/9897), 3.19 MiB | 8.23 MiB/s, done.
Resolving deltas: 100% (6636/6636), done.
Cloning into 'benchmark/googletest'...
remote: Enumerating objects: 28346, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 28346 (delta 88), reused 47 (delta 42), pack-reused 28188 (from 3)
Receiving objects: 100% (28346/28346), 13.56 MiB | 22.36 MiB/s, done.
Resolving deltas: 100% (21023/21023), done.


**Organize the code and install**

In [19]:
!rm -rf benchmark/build
!cmake -E make_directory "benchmark/build"
!cmake -E chdir "benchmark/build" cmake -DCMAKE_BUILD_TYPE=Release ..
!cmake --build "benchmark/build" --config Release --target install

-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Failed to find LLVM FileCheck
-- Found Git: /usr/bin/git (found version "2.34.1")
-- Google Benchmark version: v1.9.4-61-g7b9e482e, normalized to 1.9.4.61
-- Looking for shm_open in rt
-- Looking for shm_open in rt - found
-- Performing Test HAVE_CXX_FLAG_WALL
-- Performing Test HAVE_CXX_FLAG_WALL - Success
-- Performing Test HAVE_CXX_FLAG_WEXTRA
-- Performing Test HAVE_CXX_FLAG_WEXTRA - Success
-- Performing Test HAVE_CXX_FLAG_WSHADOW
-- Performing Test HAVE_CXX_FLAG_WSHADOW - Success
-- Performing Test HAVE_CXX_FLAG_WFLOAT_EQUAL
-- Performing Test HAVE_CXX_FLAG_WFLOAT_EQUAL - Success
-- Performing Test HAVE_CXX_FLAG_WOLD_STYLE_CAST
-- Performing Test HAVE_CXX_FLAG_WOLD_STYLE_CAST - Success
-- Performing Test HAVE_CXX_FLAG_WCON

# **How a logarithmic time primality test works**

The only exact method to determine whether a number is prime or composite is to exhaustively verifing that it does not have divisiors.
This solution has a worst-case time complexity of $Θ(\sqrt n)$, which happens everytime the number passed is prime.

An interesting approach to primality testing may be to veryfing whether a number satisfies a necessary criterion for prime numbers, which can be verified in constant time, such as the Miller-Rabin primality test.
However, for a given $n$, testing every possible $a$ may lead to a complexity of $Θ(n)$, which is even worse than the naive approach. A interesting idea may be to checking a $\Theta(\log(n))$ possible $a$s. Our implementation selects $a$s as the temporany results of the recursive calculus of $a^n mod n$.

At the end:
*   if a number is prime, we will always identify it as possibly prime (Miller-Rabin is necessary to primality but not sufficient);
*   if a number is composite, we possibly will be able to detect it but even not. Moreover, we only perform the test on a restricted number of random $a$s, therefore there may be composite numbers that do not satisfy the test only with $a$s not checked by the considered execution (repeating the execution multiple times may be successful).

We can state the latter cases in a specular way:
*   if a number is detected as possibly prime, it may be a prime or a composite number that satisfies any Miller-Rabin test performed;
*   if a number is detected as composite, it is composite for sure since it has not passed at least one Miller-Rabin test.






In [83]:
%%writefile prime.cpp

#include <boost/random.hpp> //Boost random
#include <boost/multiprecision/cpp_int.hpp> //Boost multiprecision
#include <iostream>
#include <benchmark/benchmark.h> //Google Benchmark library
#include <string.h>

using namespace boost::random;
using namespace boost::multiprecision;
using namespace std;
using uint512 = number<cpp_int_backend<512, 512, unsigned_magnitude, unchecked, void>>;
using uint100000 = number<cpp_int_backend<100000, 100000, unsigned_magnitude, unchecked, void>>;

/***************************************************************************
    RANDOM NUMBERS GENERATORS : take as inputs interval bounds (included)
****************************************************************************/

uint100000 random_uint100000(uint100000  start, uint100000 end)
/**
  100000 bits precision random number generator (useful for complexity tests in function of limbs).
*/
    {
        static boost::random::mt19937_64 rng(static_cast<unsigned>(std::time(nullptr))); //Mersenne Twister 64 bits random number generator
        boost::random::uniform_int_distribution<uint100000> dist(start, end); //uniform distribution definition with beounds requested

        return dist(rng); //random generation
    }

uint512 random_uint512(uint512  start, uint512 end)
/**
  512 bits precision random number generator
*/
    {
        static boost::random::mt19937_64 rng(static_cast<unsigned>(std::time(nullptr))); //Mersenne Twister 64 bits random number generator
        boost::random::uniform_int_distribution<uint512> dist(start, end); //uniform distribution definition with beounds requested

        return dist(rng); //random generation
    }

/*********************************************************************************
                                  Primality tests
*********************************************************************************/

bool primalityTest(uint512 n)
/**
  Naive primality test for uint512 numbers (adapted from slides)
*/
    {
        if (n == 0 || n == 1)
          return false;
        if (n == 2)
          return true;
        if (n%2 == 0)
          return false;

        for (uint512 i = 3; i*i <= n; i += 2) //"i*i" instead of "\sqrt(n)"
            {
                if (n%i == 0)
                    return false;
            }

        return true;
    }
uint512 powerAdvanced(uint512 a, uint512 p, uint512 n, bool * probPrime)
/**
  power function adapted from slides
*/
    {
        if (p == 0)
            return 1;

        uint512 x = powerAdvanced(a, p>>1, n, probPrime); //division by 2 implemented as an unitary binary right shift
        uint512 result = (x*x)%n;
        if (result == 1 && x!=1 && x!=(n-1))
            *probPrime = false;

        if (p%2 == 1)
            result = (a*result)%n;

        return result;
    }
bool primalityTestAdvanced (uint512 n)
/**
  Randomized primality test adapted from slides
*/
    {
        if (n == 1 || n == 0)
          return false;
        if (n == 2)
          return true;

        uint512 a = random_uint512(2, n-1);

        bool probPrime = true;
        uint512 result = powerAdvanced(a, n-1, n, &probPrime);

        return (result == 1 && probPrime);
    }

/********************************************************************************
    Isolated multiplications with modules to test multiplications complexities
********************************************************************************/
uint512 multiplicationWithModule512 (uint512 x, uint512 n)
    {
        return ((x*x)%n);
    }
uint100000 multiplicationWithModule100000 (uint100000 x, uint100000 n)
    {
        return ((x*x)%n);
    }

/********************************************************************************
                     Utility functions for number generation
********************************************************************************/

uint100000 fromBitsToRandom100000 (unsigned n_bits, uint100000 * start, uint100000 * end)
/**
    Utility function to uint100000 expressed exactly with "n_bits" ("start" and "end" are additionally calculated for the user)
**/
    {
        uint100000 s = n_bits<2 ? 0 : uint100000(1) << (n_bits-1);
        uint100000 e = (uint100000(1) << n_bits) -1;
        uint100000 r = random_uint100000(s, e);

        if (start != nullptr)
            *start = s;
        if (end != nullptr)
            *end = e;

        return r;
    }

uint512 fromBitsToRandom (unsigned n_bits, uint512 * start, uint512 * end)
/**
    Utility function to uint512 expressed exactly with "n_bits" ("start" and "end" are additionally calculated for the user)
**/
    {
        uint512 s = n_bits<2 ? 0 : uint512(1) << (n_bits-1);
        uint512 e = (uint512(1) << n_bits) -1;
        uint512 r = random_uint512(s, e);

        if (start != nullptr)
            *start = s;
        if (end != nullptr)
            *end = e;

        return r;
    }

/****************************************************************/
/********************** CORRECTNESS TESTS ***********************/
/****************************************************************/

void randomCorrectnessTest (benchmark::State& state)
/**
    Checks whether the number generated is inside the desired range
*/
    {
        uint512 start, end;

        for (auto _: state)
            {
                uint512 r = fromBitsToRandom(state.range(0), &start, &end);

                string ris = (r < start || r > end) ? " is out of: " : " in: ";
                std::cout << r << ris << "["<<start <<";"<< end<<"]" <<std::endl;
            }
    }

//BENCHMARK(randomCorrectnessTest)->DenseRange(0, 20)->Iterations(10); //range is defined in terms of bits used to represent the generated number

void naiveCorrectnessTest (benchmark::State& state)
/**
    Detects primality of some (not even) random numbers with the naive test
*/
    {
        for (auto _: state)
            {
                short int n_bits = state.range(0);

                uint512 r;
                do
                    {
                        r = fromBitsToRandom(n_bits, nullptr, nullptr);
                    }
                while (n_bits > 2 && r%2 == 0);
                string ris = primalityTest(r) ? "prime" : "composite";
                std::cout << r << " is detected as: "<< ris <<std::endl;
            }
    }

//BENCHMARK(naiveCorrectnessTest)->DenseRange(0, 15)->Iterations(20); //we assume that these are enough tests for considering the naive tester correct

void advancedCorrectnessTest (benchmark::State& state)
/**
     Tests that randomized test can generate errors only for "prime" responses
**/
    {
        uint512 errors = 0;
        for (auto _: state)
            {
                short int n_bits = state.range(0);
                uint512 r;
                uint512 risAdvanced;

                do
                    {
                        r = fromBitsToRandom(n_bits, nullptr, nullptr);
                        risAdvanced = primalityTestAdvanced(r);
                    }
                while (risAdvanced); //wait until the answer is "composite"

                uint512 risNaive = primalityTest(r);
                if (risNaive) //the number is prime!
                    {
                        errors++; //count the errors
                        std::cout << r << " is detected as composite by the randomized test but it is not!"<<std::endl;
                    }
            }
        std::cout << "Detected errors: " << errors << std::endl;
    }

//BENCHMARK(advancedCorrectnessTest)->DenseRange(3, 70)->Iterations(1000);

void advancedDetailedCorrectnessTest (benchmark::State& state)
/**
     Calculates the average probability of "finding" an execution that performs an error
**/
    {
        int errors = 0;

        short int n_bits = state.range(0);

        uint512 risNaive;
        uint512 r;

        float sum = 0 ;
        for (auto _: state)
            {
                do
                  {
                      r = fromBitsToRandom(n_bits, nullptr, nullptr);
                      risNaive = primalityTest(r);
                  }
                while (risNaive); //wait until the generated number is composite (with prime numbers no error is possible)

                errors = 0;
                for (int i = 0 ; i < 1000 ; i++) //do some executions to do an average over the executions
                    {
                        uint512 risAdvanced = primalityTestAdvanced(r);
                        if (risAdvanced != risNaive)
                            errors++;
                    }
                sum += errors/float(10); //first divide by 1000 to calculate the probability of finding an error in 1000 iterations,
                                         //then multiply by 100 to obtain a "%" (for readability of low probabilities)
            }

        std::cout << sum/state.iterations() << " %"<<std::endl; //average among the "state.iterations()" numbers generated of every error probability
    }

// BENCHMARK(advancedDetailedCorrectnessTest)->DenseRange(3, 40)->Iterations(10000);

/*******************************************************************/
/*********************** COMPLEXITY ANALYSIS ***********************/
/*******************************************************************/

void naiveComplexityTest (benchmark::State& state)
/**
    Simple complexity test for the naive tester
**/
    {
        long n = state.range(0);
        for (auto _: state)
            {
                primalityTest(n);
            }

        state.SetComplexityN(n);
    }

/*
BENCHMARK(naiveComplexityTest)
->Arg(3)->Arg(7)->Arg(13)->Arg(31)->Arg(61)->Arg(127)->Arg(251)->Arg(509)->Arg(1021)
->Arg(2039)->Arg(4093)->Arg(8191)->Arg(16381)->Arg(32749)->Arg(65521)->Arg(131071)->Arg(262139)
->Arg(524287)->Arg(1048573)->Arg(2097143)->Arg(4194301)->Arg(8388593)->Arg(16777213)->Arg(33554393)
->Iterations(400000)->Complexity(); //better to use prime numbers since they are the worst-case
*/

void randomComplexityTest (benchmark::State& state)
/**
    Tests complexity of the random number generator (it should be constant)
**/
    {
        short int n_bits = state.range(0);
        for (auto _: state)
            {
                uint512 r = fromBitsToRandom(n_bits, nullptr, nullptr);
            }

        state.SetComplexityN(n_bits);
    }

//BENCHMARK(randomComplexityTest)->DenseRange(0, 100)->Iterations(500000)->Complexity();

void advancedComplexityTest (benchmark::State& state)
/**
    Tests complexity of the randomized tester
**/
    {
        short int n_bits = state.range(0);
        std::vector<uint512> numbers;

        int i;

        //we fill a vector of the random numbers we will test, in order to not include this step in the whole complexity measurement
        for (i = 0 ; i < 10000 ; i++)
          {
              numbers.push_back(fromBitsToRandom(n_bits, nullptr, nullptr));
          }

        i = 0;
        for (auto _: state)
            {
                benchmark::DoNotOptimize(primalityTestAdvanced(numbers[i]));
                i++;
            }

        state.SetComplexityN(n_bits);
    }

// BENCHMARK(advancedComplexityTest)->DenseRange(0, 512)->Iterations(10000)->Complexity();

void Multiplication512ComplexityTest (benchmark::State& state)
/**
    multiplicationWithModule512 test
**/
    {
        short int n_bits = state.range(0);

        uint512 x = fromBitsToRandom(n_bits, nullptr, nullptr);
        uint512 n = fromBitsToRandom(n_bits, nullptr, nullptr);

        for (auto _: state)
            {
                benchmark::DoNotOptimize(multiplicationWithModule512(x,n));
            }

        state.SetComplexityN(n_bits);
    }

// BENCHMARK(Multiplication512ComplexityTest)->DenseRange(2, 512)->Iterations(10000)->Complexity();

void Multiplication100000ComplexityTest (benchmark::State& state)
/**
    multiplicationWithModule100000 test
**/
    {
        unsigned n_bits = state.range(0);

        n_bits *= 64;
        uint100000 x = fromBitsToRandom100000(n_bits, nullptr, nullptr);
        uint100000 n = fromBitsToRandom100000(n_bits, nullptr, nullptr);

        for (auto _: state)
            {
                benchmark::DoNotOptimize(multiplicationWithModule100000(x,n));
            }

        state.SetComplexityN(n_bits);
    }

//BENCHMARK(Multiplication100000ComplexityTest)->DenseRange(2, 1560)->Iterations(100)->Complexity();

BENCHMARK_MAIN();




Overwriting prime.cpp


In [84]:
!g++ prime.cpp -O2 -std=c++11  -lbenchmark  -o primeFinder

In [86]:
!./primeFinder

The number of inputs is very large. Multiplication512ComplexityTest will be repeated at least 511 times.
2025-10-21T21:40:07+00:00
Running ./primeFinder
Run on (2 X 2200 MHz CPU s)
CPU Caches:
  L1 Data 32 KiB (x1)
  L1 Instruction 32 KiB (x1)
  L2 Unified 256 KiB (x1)
  L3 Unified 56320 KiB (x1)
Load Average: 0.52, 0.69, 0.65
-----------------------------------------------------------------------------------------------
Benchmark                                                     Time             CPU   Iterations
-----------------------------------------------------------------------------------------------
Multiplication512ComplexityTest/2/iterations:10000         41.5 ns         41.3 ns        10000
Multiplication512ComplexityTest/3/iterations:10000         45.3 ns         45.3 ns        10000
Multiplication512ComplexityTest/4/iterations:10000         42.2 ns         42.2 ns        10000
Multiplication512ComplexityTest/5/iterations:10000         41.7 ns         41.7 ns        10000